In [1]:
from IPython.core.display import display, HTML; display(HTML("<style>.container { width:100% !important; }</style>")) # make screen full width
import torch, json, random, os, re
import torch.autograd as autograd
from torch.autograd import Variable # for emb drop out
import torch.nn.functional as F # for emb drop out
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from datetime import datetime
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from collections import Counter
from ssg import syllable_tokenize as stok
from pythainlp.tokenize import syllable_tokenize as ptok
from util import word_tokenisation as wt # for evaluation testset
torch.manual_seed(0)

def addsep(obj):
    '''
    add separation between symbols so we can separate them from characters
    we want to separate symbols from characters to improve the accuracy of syllable_tokenizer
    '''
    obj = re.sub(r'([^\|])([-_/ !"#%&,:;<=>@~0-9๐๑๒๓๔๕๖๗๘๙‘’\ufeff\r\t\\\^\$\[\]\(\)\{\}\.\?\*\+\'])', r'\1|\2', obj)
    obj = re.sub(r'([^\|])([-_/ !"#%&,:;<=>@~0-9๐๑๒๓๔๕๖๗๘๙‘’\ufeff\r\t\\\^\$\[\]\(\)\{\}\.\?\*\+\'])', r'\1|\2', obj) # twice
    obj = re.sub(r'([-_/ !"#%&,:;<=>@~0-9๐๑๒๓๔๕๖๗๘๙‘’\ufeff\r\t\\\^\$\[\]\(\)\{\}\.\?\*\+\'])([^\|])', r'\1|\2', obj)
    obj = re.sub(r'([-_/ !"#%&,:;<=>@~0-9๐๑๒๓๔๕๖๗๘๙‘’\ufeff\r\t\\\^\$\[\]\(\)\{\}\.\?\*\+\'])([^\|])', r'\1|\2', obj) # twice
    obj = re.sub(r'\|{2,5}', '|', obj)
    return obj

In [4]:
# see lines with different

def see_diff(answer_text, pred_text):
    
    a = answer_text.strip('\n').split('\n')
    p = pred_text.strip('\n').split('\n')
    at,pt = '',''
    wrongcount = 0

    for i in range(len(a)):

        if len(a[i]) > 0 and a[i] != p[i]: # each line

            idxa = 0
            idxp = 0

            while idxa < len(a[i]) and idxp < len(p[i]): # each char

                if a[i][idxa] == p[i][idxp]:
                    at += a[i][idxa]
                    idxa += 1
                    pt += p[i][idxp]
                    idxp += 1
                    #print('idxa:', idxa, 'idxp:', idxp)

                elif a[i][idxa] == '|':
                    wrongcount += 1
                    at += a[i][idxa]
                    idxa += 1
                    pt += '#'
                    #print('idxa:', idxa, 'idxp:', idxp)

                else:
                    wrongcount += 1
                    at += '#'
                    pt += p[i][idxp]
                    idxp += 1
                    #print('idxa:', idxa, 'idxp:', idxp)

            at += '\n'
            pt += '\n'


    print('wrong count =', wrongcount)
    la = at.strip('\n').split('\n')
    lp = pt.strip('\n').split('\n')

    return la, lp, at, pt

In [2]:
# find all word from train data

words = []
pat = re.compile('<[^>]{0,6}>')

for directory in ['train']:
    
    filenames = os.listdir('dataset/' + directory)
    for i, filename in enumerate(filenames):

        with open ('dataset/' + directory + '/' + filename) as file: 
            obj = file.read()
            obj = re.sub(pat, '', obj)
            lines = obj.split('\n')

        for line in lines:
            ws = line.strip('|').split('|')
            words += ws

words = list(set(words))
len(words)

74909

In [3]:
def preprocessing(txt: str, remove_space: bool = True) -> str:
    
    SEPARATOR = "|"
    SURROUNDING_SEPS_RX = re.compile("{sep}? ?{sep}$".format(sep=re.escape(SEPARATOR)))
    MULTIPLE_SEPS_RX = re.compile("{sep}+".format(sep=re.escape(SEPARATOR)))
    TAG_RX = re.compile("<\/?[A-Z]+>")
    TAILING_SEP_RX = re.compile("{sep}$".format(sep=re.escape(SEPARATOR)))

    txt = re.sub(SURROUNDING_SEPS_RX, "", txt)
    if remove_space:
        txt = re.sub("\s+", "", txt)
    txt = re.sub(MULTIPLE_SEPS_RX, SEPARATOR, txt)
    txt = re.sub(TAG_RX, "", txt)
    txt = re.sub(TAILING_SEP_RX, "", txt).strip()

    return txt

#### sy_12345

In [73]:
with open ('dataset_try/predtext.txt') as file:
    pred_text = file.read()
    lines_pred = pred_text.strip().split('\n')
with open ('dataset_try/answertext.txt') as file:
    answer_text = file.read()
    lines_answer = answer_text.strip().split('\n')
len(lines_pred), len(lines_answer)

(1, 1)

In [75]:
_, _, at, pt = see_diff(answer_text, pred_text)
len(at), len(pt)

wrong count = 3999


(549092, 549092)

In [77]:
begin, end = 0, 0
iscorrect = False
cws = []

for i in range(len(at)): #len(at)
    if at[i] == '|' and pt[i] == '|':
        end = i
        if iscorrect == True:
            cws.append(at[begin+1:end])
        else:
            iscorrect = True
        begin = i
    elif at[i] == '|' or pt[i] == '|':
        iscorrect = False

In [78]:
print('number of correct words        =', len(cws))
answer_words = answer_text.split('|')
print('number of words in answer_text =', len(answer_words))
print(f'ratio                          = {len(cws) / len(answer_words) * 100: .2f} %')

number of correct words        = 105032
number of words in answer_text = 109282
ratio                          =  96.11 %


In [79]:
answer_set = set(answer_words)
cw_set = set(cws)
words_set = set(words)
oov = answer_set - words_set 

answer_words_oov = [w for w in answer_words if w in oov]
cws_oov = [w for w in cws if w in oov]
print('number of correct words which is oov        =', len(cws_oov))
print('number of words in answer_text which is oov =', len(answer_words_oov))
print(f'recall                                      = {len(cws_oov) / len(answer_words_oov) * 100 :.2f} %')

number of correct words which is oov        = 1929
number of words in answer_text which is oov = 3137
recall                                      = 61.49 %


In [80]:
oov = sorted(list(oov))
for i,x in enumerate(oov):
    if x == 'กฏบัตร':
        print(i)
oov = oov[302:]
oov = oov[:-1]

302


In [81]:
answer_words_oov = [w for w in answer_words if w in oov]
cws_oov = [w for w in cws if w in oov]
print('number of correct words which is oov        =', len(cws_oov))
print('number of words in answer_text which is oov =', len(answer_words_oov))
print(f'recall                                      = {len(cws_oov) / len(answer_words_oov) * 100 :.2f} %')

number of correct words which is oov        = 1649
number of words in answer_text which is oov = 2790
recall                                      = 59.10 %


#### best_test_cnn.txt

In [23]:
with open ('dataset_try/best-test-cnn.txt') as file:
    pred_text = file.read()
    lines_pred = pred_text.strip().split('\n')
    new = ''
    for line in lines_pred:
        new += line + '|\n'
    pred_text = new
    lines_pred = pred_text.strip().split('\n')

with open('dataset_try/label.txt') as file: 
    answer_text = file.read() # answer text
    answer_text = re.sub(r'<[^>]{0,6}>', '', answer_text) # remove tags
    lines_answer = answer_text.strip().split('\n')

len(lines_pred), len(lines_answer)

(2252, 2252)

In [25]:
pred_text = preprocessing(pred_text)
answer_text = preprocessing(answer_text)
len(lines_pred), len(lines_answer)

(1, 1)

In [26]:
_, _, at, pt = see_diff(answer_text, pred_text)
len(at), len(pt)

wrong count = 4397


(548858, 548858)

In [28]:
begin, end = 0, 0
iscorrect = False
cws = []

for i in range(len(at)): #len(at)
    if at[i] == '|' and pt[i] == '|':
        end = i
        if iscorrect == True:
            cws.append(at[begin+1:end])
        else:
            iscorrect = True
        begin = i
    elif at[i] == '|' or pt[i] == '|':
        iscorrect = False

In [29]:
print('number of correct words        =', len(cws))
answer_words = answer_text.split('|')
print('number of words in answer_text =', len(answer_words))
print(f'ratio                          = {len(cws) / len(answer_words) * 100: .2f} %')

number of correct words        = 104169
number of words in answer_text = 109279
ratio                          =  95.32 %


In [30]:
answer_set = set(answer_words)
cw_set = set(cws)
words_set = set(words)
oov = answer_set - words_set 
len(oov)

answer_words_oov = [w for w in answer_words if w in oov]
cws_oov = [w for w in cws if w in oov]
print('number of correct words which is oov        =', len(cws_oov))
print('number of words in answer_text which is oov =', len(answer_words_oov))
print(f'recall                                      = {len(cws_oov) / len(answer_words_oov) * 100 :.2f} %')

number of correct words which is oov        = 1924
number of words in answer_text which is oov = 3137
recall                                      = 61.33 %


In [36]:
oov = sorted(list(oov))
for i,x in enumerate(oov):
    if x == 'กฏบัตร':
        print(i)
oov = oov[302:-1]

302


In [37]:
answer_words_oov = [w for w in answer_words if w in oov]
cws_oov = [w for w in cws if w in oov]
print('number of correct words which is oov        =', len(cws_oov))
print('number of words in answer_text which is oov =', len(answer_words_oov))
print(f'recall                                      = {len(cws_oov) / len(answer_words_oov) * 100 :.2f} %')

number of correct words which is oov        = 1669
number of words in answer_text which is oov = 2790
recall                                      = 59.82 %


#### best-test-bilstm-schemeb.txt

In [38]:
with open ('dataset_try/best-test-bilstm-schemeb.txt') as file:
    pred_text = file.read()
    lines_pred = pred_text.strip().split('\n')
    new = ''
    for line in lines_pred:
        new += line + '|\n'
    pred_text = new
    lines_pred = pred_text.strip().split('\n')

with open('dataset_try/label.txt') as file: 
    answer_text = file.read() # answer text
    answer_text = re.sub(r'<[^>]{0,6}>', '', answer_text) # remove tags
    lines_answer = answer_text.strip().split('\n')

len(lines_pred), len(lines_answer)

(2252, 2252)

In [40]:
pred_text = preprocessing(pred_text)
answer_text = preprocessing(answer_text)
len(lines_pred), len(lines_answer)

(1, 1)

In [41]:
_, _, at, pt = see_diff(answer_text, pred_text)
len(at), len(pt)

wrong count = 4509


(548793, 548793)

In [43]:
begin, end = 0, 0
iscorrect = False
cws = []

for i in range(len(at)): #len(at)
    if at[i] == '|' and pt[i] == '|':
        end = i
        if iscorrect == True:
            cws.append(at[begin+1:end])
        else:
            iscorrect = True
        begin = i
    elif at[i] == '|' or pt[i] == '|':
        iscorrect = False

In [44]:
print('number of correct words        =', len(cws))
answer_words = answer_text.split('|')
print('number of words in answer_text =', len(answer_words))
print(f'ratio                          = {len(cws) / len(answer_words) * 100: .2f} %')

number of correct words        = 103777
number of words in answer_text = 109279
ratio                          =  94.97 %


In [45]:
answer_set = set(answer_words)
cw_set = set(cws)
words_set = set(words)
oov = answer_set - words_set 
len(oov)

answer_words_oov = [w for w in answer_words if w in oov]
cws_oov = [w for w in cws if w in oov]
print('number of correct words which is oov        =', len(cws_oov))
print('number of words in answer_text which is oov =', len(answer_words_oov))
print(f'recall                                      = {len(cws_oov) / len(answer_words_oov) * 100 :.2f} %')

number of correct words which is oov        = 1816
number of words in answer_text which is oov = 3137
recall                                      = 57.89 %


In [49]:
oov = sorted(list(oov))
for i,x in enumerate(oov):
    if x == 'กฏบัตร':
        print(i)
oov = oov[302:-1]

302


In [50]:
answer_words_oov = [w for w in answer_words if w in oov]
cws_oov = [w for w in cws if w in oov]
print('number of correct words which is oov        =', len(cws_oov))
print('number of words in answer_text which is oov =', len(answer_words_oov))
print(f'recall                                      = {len(cws_oov) / len(answer_words_oov) * 100 :.2f} %')

number of correct words which is oov        = 1624
number of words in answer_text which is oov = 2790
recall                                      = 58.21 %


#### input_tokenised-deepcut-deepcut.txt

In [51]:
with open ('dataset_try/input_tokenised-deepcut-deepcut.txt') as file:
    pred_text = file.read()
    lines_pred = pred_text.strip().split('\n')
    new = ''
    for line in lines_pred:
        new += line + '|\n'
    pred_text = new
    lines_pred = pred_text.strip().split('\n')

with open('dataset_try/label.txt') as file: 
    answer_text = file.read() # answer text
    answer_text = re.sub(r'<[^>]{0,6}>', '', answer_text) # remove tags
    lines_answer = answer_text.strip().split('\n')

len(lines_pred), len(lines_answer)

(2252, 2252)

In [53]:
pred_text = preprocessing(pred_text)
answer_text = preprocessing(answer_text)
len(lines_pred), len(lines_answer)

(1, 1)

In [54]:
_, _, at, pt = see_diff(answer_text, pred_text)
len(at), len(pt)

wrong count = 4604


(549318, 549318)

In [56]:
begin, end = 0, 0
iscorrect = False
cws = []

for i in range(len(at)): #len(at)
    if at[i] == '|' and pt[i] == '|':
        end = i
        if iscorrect == True:
            cws.append(at[begin+1:end])
        else:
            iscorrect = True
        begin = i
    elif at[i] == '|' or pt[i] == '|':
        iscorrect = False

In [57]:
print('number of correct words        =', len(cws))
answer_words = answer_text.split('|')
print('number of words in answer_text =', len(answer_words))
print(f'ratio                          = {len(cws) / len(answer_words) * 100: .2f} %')

number of correct words        = 104090
number of words in answer_text = 109279
ratio                          =  95.25 %


In [58]:
answer_set = set(answer_words)
cw_set = set(cws)
words_set = set(words)
oov = answer_set - words_set 
len(oov)

answer_words_oov = [w for w in answer_words if w in oov]
cws_oov = [w for w in cws if w in oov]
print('number of correct words which is oov        =', len(cws_oov))
print('number of words in answer_text which is oov =', len(answer_words_oov))
print(f'recall                                      = {len(cws_oov) / len(answer_words_oov) * 100 :.2f} %')

number of correct words which is oov        = 1836
number of words in answer_text which is oov = 3137
recall                                      = 58.53 %


In [60]:
oov = sorted(list(oov))
for i,x in enumerate(oov):
    if x == 'กฏบัตร':
        print(i)
oov = oov[302:-1]

302


In [61]:
answer_words_oov = [w for w in answer_words if w in oov]
cws_oov = [w for w in cws if w in oov]
print('number of correct words which is oov        =', len(cws_oov))
print('number of words in answer_text which is oov =', len(answer_words_oov))
print(f'recall                                      = {len(cws_oov) / len(answer_words_oov) * 100 :.2f} %')

number of correct words which is oov        = 1559
number of words in answer_text which is oov = 2790
recall                                      = 55.88 %


#### input_tokenised-pythainlp-newmm.txt

In [62]:
with open ('dataset_try/input_tokenised-pythainlp-newmm.txt') as file:
    pred_text = file.read()
    lines_pred = pred_text.strip().split('\n')
    new = ''
    for line in lines_pred:
        new += line + '|\n'
    pred_text = new
    lines_pred = pred_text.strip().split('\n')

with open('dataset_try/label.txt') as file: 
    answer_text = file.read() # answer text
    answer_text = re.sub(r'<[^>]{0,6}>', '', answer_text) # remove tags
    lines_answer = answer_text.strip().split('\n')

len(lines_pred), len(lines_answer)

(2252, 2252)

In [64]:
pred_text = preprocessing(pred_text)
answer_text = preprocessing(answer_text)
lines_pred = pred_text.strip().split('\n')
lines_answer = answer_text.strip().split('\n')
len(lines_pred), len(lines_answer)

(1, 1)

In [65]:
_, _, at, pt = see_diff(answer_text, pred_text)
len(at), len(pt)

wrong count = 25673


(553627, 553627)

In [67]:
begin, end = 0, 0
iscorrect = False
cws = []

for i in range(len(at)): #len(at)
    if at[i] == '|' and pt[i] == '|':
        end = i
        if iscorrect == True:
            cws.append(at[begin+1:end])
        else:
            iscorrect = True
        begin = i
    elif at[i] == '|' or pt[i] == '|':
        iscorrect = False

In [68]:
print('number of correct words        =', len(cws))
answer_words = answer_text.split('|')
print('number of words in answer_text =', len(answer_words))
print(f'ratio                          = {len(cws) / len(answer_words) * 100: .2f} %')

number of correct words        = 70927
number of words in answer_text = 109279
ratio                          =  64.90 %


In [69]:
answer_set = set(answer_words)
cw_set = set(cws)
words_set = set(words)
oov = answer_set - words_set 
len(oov)

answer_words_oov = [w for w in answer_words if w in oov]
cws_oov = [w for w in cws if w in oov]
print('number of correct words which is oov        =', len(cws_oov))
print('number of words in answer_text which is oov =', len(answer_words_oov))
print(f'recall                                      = {len(cws_oov) / len(answer_words_oov) * 100 :.2f} %')

number of correct words which is oov        = 709
number of words in answer_text which is oov = 3137
recall                                      = 22.60 %


In [71]:
oov = sorted(list(oov))
for i,x in enumerate(oov):
    if x == 'กฏบัตร':
        print(i)
oov = oov[302:-1]

302


In [72]:
answer_words_oov = [w for w in answer_words if w in oov]
cws_oov = [w for w in cws if w in oov]
print('number of correct words which is oov        =', len(cws_oov))
print('number of words in answer_text which is oov =', len(answer_words_oov))
print(f'recall                                      = {len(cws_oov) / len(answer_words_oov) * 100 :.2f} %')

number of correct words which is oov        = 433
number of words in answer_text which is oov = 2790
recall                                      = 15.52 %


#### results

In [ ]:
# input_tokenised-pythainlp-newmm.txt
number of correct words which is oov        = 433
number of words in answer_text which is oov = 2790
recall                                      = 15.52 %

# input_tokenised-deepcut-deepcut.txt
number of correct words which is oov        = 1559
number of words in answer_text which is oov = 2790
recall                                      = 55.88 %

# best-test-bilstm-schemeb.txt
number of correct words which is oov        = 1624
number of words in answer_text which is oov = 2790
recall                                      = 58.21 %

# syllable model scheme C
number of correct words which is oov        = 1649
number of words in answer_text which is oov = 2790
recall                                      = 59.10 %

# best_test_cnn.txt
number of correct words which is oov        = 1669
number of words in answer_text which is oov = 2790
recall                                      = 59.82 %